# RULER on clinical text: Bio_ClinicalBERT

This notebook is the clinical-text diagnostic of **Section 5.5** of the RULER paper.
It applies the same $M_2$ and $M_4$ metrics used for the tabular MLPs to a
transformer trained with a **non-classification objective** — masked language
modelling — to show that RULER depends only on the geometry of a penultimate
representation, not on the training task.

**What is being tested.** Bio_ClinicalBERT is fine-tuned on clinical notes, a
forget set is defined at the document level, four approximate unlearning methods
are applied, and both metrics are computed. The pre-unlearning $M_4$ diagnostic
asks a prior question: were these documents memorised at all?

**Metric code is imported, not reimplemented.** `m2` and `m4` come from the
`ruler` package, the same functions the tabular experiments use. That is the
claim being demonstrated — the metrics apply to this setting *without
modification* — so reimplementing them here would undercut it.

**Representation.** The `[CLS]` activation of the final transformer layer
(768-d), the analogue of the tabular MLP's penultimate ReLU: the last
whole-sequence abstraction before the task head (Appendix Table 2, Fig. 10).

**Expected result** (paper Section 5.5, Fig. 4). Pre-unlearning $M_4$ is 0.537,
0.521 and 0.501 at ff = 1%, 5%, 10%: near the null of 0.50, indicating weak or
no detectable representation-level memorisation. $M_2$ stays small at 1% and
becomes more variable at larger fractions, taking **both signs** across methods.
Unlike the tabular setting, there is no consistent negative residual here — with
little memorisation to remove, unlearning mostly perturbs the geometry.

> **Runtime.** Fine-tuning BERT for 3 forget fractions x 5 seeds is expensive.
> On a single GPU expect a few hours; on CPU it is impractical. Checkpoints are
> cached to `MODELS_DIR`, so an interrupted run resumes.

## 1. Setup

In [ ]:
import copy
import os
import sys
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import optim
from transformers import AutoModelForMaskedLM, AutoTokenizer

# The metrics and statistics come from the ruler package at the repository root.
sys.path.insert(0, os.path.abspath('..'))
from ruler import m2, m4
from paper.stats import significance_stars, wilcoxon_signed_rank

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cpu':
    print('WARNING: fine-tuning BERT on CPU is impractical. Reduce NUM_SEEDS and '
          'MAX_SENTENCES_PER_EPOCH for a smoke test.')

In [ ]:
MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'

# --- Experiment grid (paper Section 5.5) ---------------------------------
FORGET_FRACTIONS = [0.01, 0.05, 0.10]   # matches Fig. 4
NUM_SEEDS        = 5                    # BERT fine-tuning is expensive

# --- Representation --------------------------------------------------------
MAX_SEQ_LEN = 128
EMB_DIM     = 768                       # BERT-base hidden size

# --- Fine-tuning (original model and oracle) -------------------------------
FINETUNE_EPOCHS = 3
LR_FINETUNE     = 2e-5                  # standard BERT fine-tuning rate
BATCH_SIZE      = 16
MASK_PROB       = 0.15                  # BERT's masking rate

# Sentences sampled per epoch. Fine-tuning on every sentence of every note is
# unnecessary for this diagnostic and dominates runtime; the same budget is
# used for the original model and the oracle so neither is advantaged.
MAX_SENTENCES_PER_EPOCH = 1000

# --- Unlearning (mirrors the tabular hyperparameters, Section 4.2) ---------
LR_UNLEARN   = 5e-5
GA_EPOCHS    = 5
UL_EPOCHS    = 10
ALPHA        = 0.6
TEMPERATURE  = 2.0
UL_SEED      = 100                      # fixed unlearning seed
FORGET_SEED  = 999                      # forget-set sampling, as in the paper

# --- Metric evaluation (Section 4.3) --------------------------------------
MAX_RETAIN_CAL = 500                    # M2 calibration subsample
MAX_RETAIN_GEO = 2000                   # M4 gallery cap
SUBSAMPLE_SEED = 42

METHODS = ['Gradient Ascent', 'NegGrad+', 'Fine-Tuning', 'SCRUB']

MODELS_DIR = './clinical_models/'
FIGS_DIR   = './clinical_figures/'
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

print(f'Model {MODEL_NAME}')
print(f'Forget fractions {FORGET_FRACTIONS}, {NUM_SEEDS} seeds')

## 2. Data

Two freely available clinical-text sources:

- **MTSamples** — medical transcription notes, no licence required.
  Download from
  [Kaggle](https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions)
  and set `MTSAMPLES_PATH`. This is the source reported in the paper.
- **i2b2 2014** — de-identification corpus, requires registration at
  [DBMI](https://portal.dbmi.hms.harvard.edu). Set `I2B2_DIR` to the directory
  of `.txt` files.

Notes are split into sentences; each note is one **document**, which is the unit
of erasure. A GDPR request removes an individual's whole record, not a random
sample of their sentences, so the forget set is defined at document level and
then expanded to sentences.

In [ ]:
import re

I2B2_DIR       = './data/i2b2_2014/'
MTSAMPLES_PATH = './data/mtsamples.csv'

MIN_SENT_CHARS, MAX_SENT_CHARS = 20, 300


def split_sentences(text):
    """Split a clinical note into sentences, filtered by character length.

    Very short fragments (headings, list bullets) carry little context for MLM,
    and very long ones would be truncated at MAX_SEQ_LEN anyway.
    """
    parts = re.split(r'(?<=[.!?;])\s+|\n+', str(text))
    return [p.strip() for p in parts
            if MIN_SENT_CHARS <= len(p.strip()) <= MAX_SENT_CHARS]


def load_i2b2(data_dir=I2B2_DIR):
    """One .txt file per patient record."""
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f'i2b2 directory not found at {data_dir}')
    records = []
    for i, name in enumerate(sorted(os.listdir(data_dir))):
        if name.endswith('.txt'):
            with open(os.path.join(data_dir, name), errors='ignore') as handle:
                records.append({'doc_id': i, 'text': handle.read()})
    if not records:
        raise FileNotFoundError(f'no .txt files in {data_dir}')
    return pd.DataFrame(records)


def load_mtsamples(path=MTSAMPLES_PATH):
    """One transcription per row; each row is treated as one document."""
    if not os.path.exists(path):
        raise FileNotFoundError(f'MTSamples CSV not found at {path}')
    frame = pd.read_csv(path).dropna(subset=['transcription'])
    frame = frame.rename(columns={'transcription': 'text'})
    frame['doc_id'] = range(len(frame))
    return frame[['doc_id', 'text']]


def build_sentence_frame(note_frame):
    rows = [{'doc_id': row.doc_id, 'sentence': sentence}
            for row in note_frame.itertuples()
            for sentence in split_sentences(row.text)]
    return pd.DataFrame(rows)


try:
    notes = load_i2b2()
    DATA_SOURCE = 'i2b2-2014'
except FileNotFoundError as error:
    print(f'i2b2 unavailable ({error}); using MTSamples.')
    notes = load_mtsamples()
    DATA_SOURCE = 'MTSamples'

sent_df = build_sentence_frame(notes)
print(f'{DATA_SOURCE}: {sent_df["doc_id"].nunique()} documents, '
      f'{len(sent_df)} sentences')

## 3. Encoder and representation extraction

A single `BertForMaskedLM` serves both roles: its **MLM head** provides the
training objective, and its **encoder** provides the `[CLS]` representation the
metrics consume.

This matters. The original version of this notebook wrapped `AutoModel`, which
is a bare `BertModel` with no MLM head, and then called it with `labels=` —
a `TypeError`, so no training could run. Using `AutoModelForMaskedLM` gives
one model that can both be trained and be measured.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ClinicalBERT(torch.nn.Module):
    """Bio_ClinicalBERT with an MLM head, plus [CLS] representation access."""

    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.mlm = AutoModelForMaskedLM.from_pretrained(model_name)

    def forward(self, input_ids, attention_mask, labels=None):
        """MLM forward pass. Returns an object with .loss and .logits."""
        return self.mlm(input_ids=input_ids, attention_mask=attention_mask,
                        labels=labels)

    def embed(self, input_ids, attention_mask):
        """[CLS] activation of the final transformer layer, shape (B, 768)."""
        hidden = self.mlm.bert(input_ids=input_ids,
                               attention_mask=attention_mask).last_hidden_state
        return hidden[:, 0, :]


def encode(sentences, max_len=MAX_SEQ_LEN):
    return tokenizer(list(sentences), padding=True, truncation=True,
                     max_length=max_len, return_tensors='pt').to(DEVICE)


@torch.no_grad()
def representations(model, sentences, batch_size=64):
    """[CLS] embeddings for a list of sentences, as an (N, 768) array.

    Evaluation mode is forced so dropout is inactive: the metrics compare a
    record's embedding across models, which requires it to be deterministic.
    L2 normalisation is left to the metric functions, which apply it
    themselves.
    """
    was_training = model.training
    model.eval()
    chunks = []
    for start in range(0, len(sentences), batch_size):
        batch = encode(sentences[start:start + batch_size])
        chunks.append(model.embed(batch['input_ids'],
                                  batch['attention_mask']).cpu().numpy())
    if was_training:
        model.train()
    return np.concatenate(chunks, axis=0) if chunks else np.empty((0, EMB_DIM))


# Sanity check: correct shape, and the model runs end to end.
_probe = ClinicalBERT().to(DEVICE)
_out = representations(_probe, ['The patient presented with chest pain.',
                                'Blood pressure was 130/85 on admission.'])
assert _out.shape == (2, EMB_DIM), _out.shape
print(f'Encoder ready; representation shape {_out.shape}')
del _probe, _out

## 4. Document-level forget/retain partition

The forget set is all sentences belonging to a randomly chosen subset of
documents, sampled once at a fixed random state (999) so that all four
unlearning methods and all seeds erase exactly the same records.

In [ ]:
def partition_documents(frame, forget_fraction, seed=FORGET_SEED):
    """Split sentences into retain and forget sets by document.

    Selection is at document level, then expanded to sentences, so no document
    is split across the two sets — a partial erasure would leave the very
    residuals the metrics are meant to detect.
    """
    documents = np.sort(frame['doc_id'].unique())
    n_forget = max(1, int(np.floor(forget_fraction * len(documents))))
    rng = np.random.RandomState(seed)
    forget_docs = set(rng.choice(documents, n_forget, replace=False))

    is_forget = frame['doc_id'].isin(forget_docs).to_numpy()
    sentences = frame['sentence'].tolist()
    forget = [s for s, flag in zip(sentences, is_forget, strict=True) if flag]
    retain = [s for s, flag in zip(sentences, is_forget, strict=True) if not flag]
    if not retain:
        raise ValueError(
            f'forget fraction {forget_fraction:.0%} selected all {len(documents)} '
            'document(s), leaving no retain set. The metrics need retained '
            'records to compare against; use more documents or a smaller fraction.'
        )
    return {'forget': forget, 'retain': retain, 'all': sentences,
            'n_forget_docs': n_forget, 'n_docs': len(documents)}


print(f'{"ff":>5}  {"docs":>6}  {"forget sents":>13}  {"retain sents":>13}')
for ff in FORGET_FRACTIONS:
    part = partition_documents(sent_df, ff)
    print(f'{ff:>5.0%}  {part["n_forget_docs"]:>6}  '
          f'{len(part["forget"]):>13}  {len(part["retain"]):>13}')

## 5. Fine-tuning the original model and the oracle

- **Original** $\theta^o$: Bio_ClinicalBERT fine-tuned on **all** sentences.
- **Oracle** $\theta^r$: Bio_ClinicalBERT fine-tuned on the **retain** sentences
  only.

Both start from the *pretrained* checkpoint with the same seed. The oracle must
never be derived from the original model: it is defined as a model with no
knowledge of the forget set, and copying the original's weights would carry
that knowledge straight into the reference the metrics are calibrated against.

In [ ]:
def mlm_batch_loss(model, sentences, generator, mask_prob=MASK_PROB):
    """Masked-language-modelling loss for a batch of sentences."""
    batch = encode(sentences)
    input_ids = batch['input_ids'].clone()
    labels = input_ids.clone()

    # Never mask special tokens or padding; both would be free to predict.
    special = torch.zeros_like(labels, dtype=torch.bool)
    for token_id in tokenizer.all_special_ids:
        special |= labels == token_id

    probabilities = torch.full(labels.shape, mask_prob, device=labels.device)
    probabilities[special] = 0.0
    masked = torch.bernoulli(probabilities, generator=generator).bool()

    if not masked.any():
        # A Bernoulli draw over few maskable tokens can select none, and the
        # loss is then a mean over zero elements: NaN. One NaN gradient
        # poisons every weight permanently, with no error raised. Small final
        # batches hit this a few percent of the time, so force one position.
        candidates = ~special & batch['attention_mask'].bool()
        if not candidates.any():
            return torch.zeros((), device=labels.device, requires_grad=True)
        scores = torch.rand(labels.shape, device=labels.device, generator=generator)
        scores[~candidates] = -1.0
        masked.view(-1)[int(scores.argmax())] = True

    labels[~masked] = -100          # loss only on masked positions
    input_ids[masked] = tokenizer.mask_token_id
    return model(input_ids, batch['attention_mask'], labels=labels).loss


def finetune(model, sentences, epochs=FINETUNE_EPOCHS, lr=LR_FINETUNE, seed=0):
    """Fine-tune with the MLM objective on the given sentences."""
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    rng = np.random.RandomState(seed)
    optimiser = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    model.train()

    for epoch in range(epochs):
        n_sample = min(MAX_SENTENCES_PER_EPOCH, len(sentences))
        sample = [sentences[i] for i in
                  rng.choice(len(sentences), n_sample, replace=False)]
        total, steps = 0.0, 0
        for start in range(0, len(sample), BATCH_SIZE):
            optimiser.zero_grad()
            loss = mlm_batch_loss(model, sample[start:start + BATCH_SIZE], generator)
            loss.backward()
            optimiser.step()
            total += loss.item()
            steps += 1
        print(f'      epoch {epoch + 1}/{epochs}  mean loss {total / max(steps, 1):.4f}')

    model.eval()
    return model


def checkpoint(ff, seed, kind):
    """Checkpoint path for a fine-tuned model.

    The original model is fine-tuned on *all* sentences, so it is identical for
    every forget fraction; its filename therefore omits ff and the same file is
    reused across fractions. Keying it by fraction would silently fine-tune and
    store the same BERT three times. Oracles genuinely differ per fraction.
    """
    if kind == 'orig':
        return os.path.join(MODELS_DIR, f'clinbert_s{seed}_orig.pt')
    return os.path.join(MODELS_DIR, f'clinbert_ff{round(ff * 100):02d}_s{seed}_{kind}.pt')


def load_or_finetune(path, sentences, seed):
    """Load a cached fine-tuned model, or fine-tune from pretrained and cache it."""
    model = ClinicalBERT().to(DEVICE)
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
        model.eval()
        return model, True
    torch.manual_seed(seed)
    finetune(model, sentences, seed=seed)
    torch.save(model.state_dict(), path)
    return model, False


print('Fine-tuning utilities ready.')

## 6. Unlearning methods

The same four methods as the tabular experiments (Section 4.2), with the MLM
loss replacing cross-entropy as the natural objective for this model class:

| Method | Objective | Epochs |
|---|---|---|
| Gradient Ascent | maximise MLM loss on the forget set | 5 |
| NegGrad+ | $\alpha\,\mathcal{L}_{\text{retain}} - (1-\alpha)\,\mathcal{L}_{\text{forget}}$ | 10 |
| Fine-Tuning | MLM loss on the retain set only | 10 |
| SCRUB | KL to a frozen teacher: agree on retain, disagree on forget | 10 |

**SCRUB operates on output distributions, not representations.** An earlier
version of this notebook implemented SCRUB as cosine similarity between student
and teacher `[CLS]` embeddings. That makes the evaluation circular: the method
would be directly optimising the very geometry $M_2$ and $M_4$ measure, so a
result either way would say nothing about unlearning. Distilling over the MLM
vocabulary distribution keeps the objective at the output level, as in the
tabular implementation and in Kurmanji et al.

In [ ]:
def _fresh(model, seed):
    """A seeded working copy of the original model, in training mode."""
    torch.manual_seed(seed)
    student = copy.deepcopy(model).to(DEVICE)
    student.train()
    return student, torch.Generator(device=DEVICE).manual_seed(seed)


def _sample(sentences, n, rng):
    n = min(n, len(sentences))
    return [sentences[i] for i in rng.choice(len(sentences), n, replace=False)]


def ga_unlearn(model, forget, seed=UL_SEED):
    """Gradient ascent: maximise MLM loss on the forget set."""
    student, generator = _fresh(model, seed)
    rng = np.random.RandomState(seed)
    optimiser = optim.AdamW(student.parameters(), lr=LR_UNLEARN, weight_decay=0.01)
    for _ in range(GA_EPOCHS):
        batch_pool = _sample(forget, 200, rng)
        for start in range(0, len(batch_pool), BATCH_SIZE):
            optimiser.zero_grad()
            loss = mlm_batch_loss(student, batch_pool[start:start + BATCH_SIZE], generator)
            (-loss).backward()      # ascent
            optimiser.step()
    student.eval()
    return student


def neggrad_plus_unlearn(model, forget, retain, seed=UL_SEED):
    """Retain descent weighted by alpha, forget ascent by (1 - alpha)."""
    student, generator = _fresh(model, seed)
    rng = np.random.RandomState(seed)
    optimiser = optim.AdamW(student.parameters(), lr=LR_UNLEARN, weight_decay=0.01)
    for _ in range(UL_EPOCHS):
        retain_pool = _sample(retain, 500, rng)
        forget_pool = _sample(forget, 100, rng)
        for start in range(0, len(retain_pool), BATCH_SIZE):
            retain_batch = retain_pool[start:start + BATCH_SIZE]
            # Cycle through the forget pool so every step sees forget records,
            # even though it is smaller than the retain pool.
            offset = (start // BATCH_SIZE * BATCH_SIZE) % max(len(forget_pool), 1)
            forget_batch = forget_pool[offset:offset + BATCH_SIZE] or forget_pool[:BATCH_SIZE]
            optimiser.zero_grad()
            loss = (ALPHA * mlm_batch_loss(student, retain_batch, generator)
                    - (1 - ALPHA) * mlm_batch_loss(student, forget_batch, generator))
            loss.backward()
            optimiser.step()
    student.eval()
    return student


def finetune_unlearn(model, retain, seed=UL_SEED):
    """Continued MLM training on the retain set; erasure by catastrophic forgetting."""
    student, generator = _fresh(model, seed)
    rng = np.random.RandomState(seed)
    optimiser = optim.AdamW(student.parameters(), lr=LR_UNLEARN, weight_decay=0.01)
    for _ in range(UL_EPOCHS):
        pool = _sample(retain, 500, rng)
        for start in range(0, len(pool), BATCH_SIZE):
            optimiser.zero_grad()
            mlm_batch_loss(student, pool[start:start + BATCH_SIZE], generator).backward()
            optimiser.step()
    student.eval()
    return student


def _distillation_kl(student_logits, teacher_logits, attention_mask, temperature=TEMPERATURE):
    """KL(teacher || student) over the vocabulary, averaged across real tokens.

    Padding is excluded: it carries no information and, on a heavily padded
    batch, would dominate the average.
    """
    student_log_p = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_p = F.softmax(teacher_logits / temperature, dim=-1)
    per_token = F.kl_div(student_log_p, teacher_p, reduction='none').sum(-1)
    mask = attention_mask.bool()
    return (per_token[mask].mean()) * temperature ** 2


def scrub_unlearn(model, forget, retain, seed=UL_SEED):
    """Distil from the frozen original: agree on retain, disagree on forget."""
    student, _ = _fresh(model, seed)
    teacher = copy.deepcopy(model).to(DEVICE).eval()
    for parameter in teacher.parameters():
        parameter.requires_grad_(False)

    rng = np.random.RandomState(seed)
    optimiser = optim.AdamW(student.parameters(), lr=LR_UNLEARN, weight_decay=0.01)

    def kl_on(sentences):
        batch = encode(sentences)
        with torch.no_grad():
            teacher_logits = teacher(batch['input_ids'], batch['attention_mask']).logits
        student_logits = student(batch['input_ids'], batch['attention_mask']).logits
        return _distillation_kl(student_logits, teacher_logits, batch['attention_mask'])

    for _ in range(UL_EPOCHS):
        retain_pool = _sample(retain, 200, rng)
        forget_pool = _sample(forget, 50, rng)
        for start in range(0, len(retain_pool), BATCH_SIZE):
            retain_batch = retain_pool[start:start + BATCH_SIZE]
            offset = (start // BATCH_SIZE * BATCH_SIZE) % max(len(forget_pool), 1)
            forget_batch = forget_pool[offset:offset + BATCH_SIZE] or forget_pool[:BATCH_SIZE]
            optimiser.zero_grad()
            loss = ALPHA * kl_on(retain_batch) - (1 - ALPHA) * kl_on(forget_batch)
            loss.backward()
            optimiser.step()
    student.eval()
    return student


def run_all_methods(model, forget, retain):
    return {
        'Gradient Ascent': ga_unlearn(model, forget),
        'NegGrad+':        neggrad_plus_unlearn(model, forget, retain),
        'Fine-Tuning':     finetune_unlearn(model, retain),
        'SCRUB':           scrub_unlearn(model, forget, retain),
    }


print(f'Unlearning ready: {METHODS}')

## 7. Metrics

$M_2$ and $M_4$ are imported from `ruler` — the same implementations
used for the tabular MLPs. Only the *inputs* differ: `[CLS]` embeddings in
place of penultimate ReLU activations.

The subsampling follows Section 4.3: $\min(500, |\mathcal{D}_r|)$ retain
records for the $M_2$ baseline, and a gallery capped at 2,000 for $M_4$.

In [ ]:
def subsample(items, cap, seed=SUBSAMPLE_SEED):
    if len(items) <= cap:
        return items
    idx = np.random.RandomState(seed).choice(len(items), cap, replace=False)
    return [items[i] for i in idx]


def compute_m2(unlearned, oracle, forget_sents, retain_sents):
    """Signed calibration gap between an unlearned model and the oracle."""
    calibration = subsample(retain_sents, MAX_RETAIN_CAL)
    gap = m2(
        representations(unlearned, forget_sents),
        representations(oracle, forget_sents),
        representations(unlearned, calibration),
        representations(oracle, calibration),
    )
    return {'m2': gap}


def compute_m4(model, forget_sents, retain_sents):
    """Oracle-free percentile rank under a single model."""
    gallery = subsample(retain_sents, MAX_RETAIN_GEO)
    return m4(representations(model, forget_sents),
              representations(model, gallery))


print('Metrics wired to the ruler library (no reimplementation).')

## 8. Main experiment loop

For each (forget fraction, seed):

1. partition documents into retain and forget,
2. fine-tune (or load) the original model on all sentences,
3. record the **pre-unlearning** $M_4$ diagnostic on the original model,
4. fine-tune (or load) the oracle on retain sentences only, from pretrained,
5. apply the four unlearning methods and compute $M_2$ and $M_4$.

In [ ]:
results = []      # one row per (ff, seed, method)
pre_rows = []     # one row per (ff, seed): the pre-unlearning diagnostic

for ff in FORGET_FRACTIONS:
    print(f'\n{"=" * 62}\nForget fraction {ff:.0%}\n{"=" * 62}')
    part = partition_documents(sent_df, ff)
    forget_sents, retain_sents, all_sents = part['forget'], part['retain'], part['all']
    print(f'  {part["n_forget_docs"]}/{part["n_docs"]} documents, '
          f'{len(forget_sents)} forget / {len(retain_sents)} retain sentences')

    for seed in range(NUM_SEEDS):
        print(f'\n  Seed {seed}')
        original, cached = load_or_finetune(checkpoint(ff, seed, 'orig'), all_sents, seed)
        print(f'    original model {"loaded from cache" if cached else "fine-tuned"}')

        pre_m4 = compute_m4(original, forget_sents, retain_sents)
        pre_rows.append({'forget_fraction': ff, 'seed': seed, 'pre_unlearning_m4': pre_m4})
        print(f'    pre-unlearning M4 = {pre_m4:.4f} (null 0.50)')

        # Oracle: fine-tuned from the pretrained checkpoint on retain only.
        oracle, cached = load_or_finetune(checkpoint(ff, seed, 'oracle'), retain_sents, seed)
        print(f'    oracle {"loaded from cache" if cached else "fine-tuned"}')

        for method, model in run_all_methods(original, forget_sents, retain_sents).items():
            row = {'forget_fraction': ff, 'seed': seed, 'method': method,
                   'pre_unlearning_m4': pre_m4}
            row.update(compute_m2(model, oracle, forget_sents, retain_sents))
            row['m4'] = compute_m4(model, forget_sents, retain_sents)
            results.append(row)
            print(f'    {method:<16} M2={row["m2"]:+.5f}  M4={row["m4"]:.4f}')
            del model

        del original, oracle
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
pre_df = pd.DataFrame(pre_rows)
results_df.to_csv(os.path.join(FIGS_DIR, 'clinical_results.csv'), index=False)
print(f'\nDone: {len(results_df)} rows written to {FIGS_DIR}clinical_results.csv')

## 9. Statistical tests

One-sample Wilcoxon signed-rank tests, $M_2$ against 0 and $M_4$ against 0.50,
using `paper.stats`.

**On statistical power.** With $N = 5$ seeds the smallest attainable two-sided
$p$-value is $2/2^5 = 0.062$, so **no condition here can reach $p < 0.05$**,
however consistent the effect. The paper states this for the five-seed
diagnostics (Appendix A.14) and reports the descriptive values instead. The
$p$-values below are shown for completeness; the direction and magnitude of the
estimates carry the finding, not their significance.

In [ ]:
MIN_P_AT_N5 = 2 / 2 ** NUM_SEEDS
print(f'N = {NUM_SEEDS} seeds; smallest attainable two-sided p is {MIN_P_AT_N5:.3f}')
print('No result below can reach p < 0.05 -- read the estimates, not the stars.\n')

summary = []
for metric, null in (('m2', 0.0), ('m4', 0.50)):
    for ff in FORGET_FRACTIONS:
        for method in METHODS:
            cell = results_df[(results_df['forget_fraction'] == ff)
                              & (results_df['method'] == method)]
            values = cell[metric].to_numpy()
            if len(values) == 0:
                continue
            test = wilcoxon_signed_rank(values, null=null, min_n=3)
            summary.append({
                'metric': metric, 'forget_fraction': ff, 'method': method,
                'n': len(values), 'mean': float(np.mean(values)),
                'sd': float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
                'p': test.p_value, 'r_rb': test.rank_biserial,
            })

summary_df = pd.DataFrame(summary)

for metric, null, label in (('m2', 0.0, 'M2 signed calibration gap'),
                            ('m4', 0.50, 'M4 percentile rank')):
    # M2 is a signed gap, so its sign carries the finding; M4 is a rank in
    # [0, 1] and reads oddly with a forced sign.
    value_format = '>+11.5f' if metric == 'm2' else '>11.4f'
    print(f'{label} (null = {null})')
    print(f'  {"ff":>5}  {"Method":<16}{"mean":>11}{"sd":>10}{"p":>8}  sig')
    for _, row in summary_df[summary_df['metric'] == metric].iterrows():
        print(f'  {row["forget_fraction"]:>5.0%}  {row["method"]:<16}'
              f'{row["mean"]:{value_format}}{row["sd"]:>10.5f}{row["p"]:>8.3f}'
              f'  {significance_stars(row["p"])}')
    print()

## 10. Pre-unlearning diagnostic

$M_4$ on the **original** model, before any unlearning. This is the diagnostic
use of Lens 2 (Section 5.5): it reports whether the forget documents were
memorised in the first place.

Values near 0.50 mean the documents already blend into the general
distribution — there is no representation-level memorisation to remove, and
applying unlearning anyway risks introducing artefacts rather than fixing
anything.

In [ ]:
print('Pre-unlearning M4 (null = 0.50)')
print(f'  {"ff":>5}{"mean":>9}{"sd":>9}   Reading')
for ff in FORGET_FRACTIONS:
    values = pre_df.loc[pre_df['forget_fraction'] == ff, 'pre_unlearning_m4'].to_numpy()
    if len(values) == 0:
        continue
    mean = float(np.mean(values))
    sd = float(np.std(values, ddof=1)) if len(values) > 1 else float('nan')
    if abs(mean - 0.50) < 0.03:
        reading = 'at the null: no detectable memorisation'
    elif mean > 0.50:
        reading = 'above the null: some residual integration'
    else:
        reading = 'below the null: forget records already atypical'
    print(f'  {ff:>5.0%}{mean:>9.4f}{sd:>9.4f}   {reading}')

print('\nPaper Section 5.5 reports 0.537, 0.521, 0.501 at ff = 1%, 5%, 10%:')
print('near the null, indicating weak or no representation-level memorisation.')
print('Compare with LFW face identity (Appendix Table 14), where pre-unlearning')
print('M4 reaches 0.94 -- strong identity-level memorisation that no tested')
print('method fully erases.')

## 11. Figure (paper Fig. 4)

In [ ]:
import matplotlib.pyplot as plt

from paper.plotting import DOUBLE_COLUMN, NULL_COLOUR, apply_style, method_colours

apply_style()
edges, fills = method_colours()      # method_colours(accessible=True) for the CVD-safe palette

fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COLUMN, 2.8))
x = np.arange(len(FORGET_FRACTIONS))
width = 0.18


def mean_of(metric, forget_fraction, method):
    """Mean value for one cell of the summary table.

    Plain boolean masking rather than DataFrame.query(): query() resolves
    @-variables from the calling frame, which does not reach enclosing-function
    locals from inside a comprehension.
    """
    mask = ((summary_df['metric'] == metric)
            & (summary_df['forget_fraction'] == forget_fraction)
            & (summary_df['method'] == method))
    return summary_df.loc[mask, 'mean'].mean()


def bars(ax, metric, null):
    for i, method in enumerate(METHODS):
        means = [mean_of(metric, ff, method) for ff in FORGET_FRACTIONS]
        ax.bar(x + i * width, means, width, label=method,
               color=fills[method], edgecolor=edges[method], linewidth=0.7)
    ax.axhline(null, color=NULL_COLOUR, linestyle='--', linewidth=0.8)
    ax.set_xticks(x + 1.5 * width)
    ax.set_xticklabels([f'{ff:.0%}' for ff in FORGET_FRACTIONS])
    ax.set_xlabel('Forget fraction')


bars(axes[0], 'm2', 0.0)
axes[0].set_ylabel('$M_2$: signed calibration gap')
axes[0].set_title('(a) Oracle-comparative $M_2$')

bars(axes[1], 'm4', 0.50)
pre_means = [pre_df.loc[pre_df['forget_fraction'] == ff, 'pre_unlearning_m4'].mean()
             for ff in FORGET_FRACTIONS]
axes[1].plot(x + 1.5 * width, pre_means, 'D', color=NULL_COLOUR, markersize=4,
             label='Pre-unlearning $M_4$', zorder=5)
axes[1].set_ylabel('$M_4$: percentile rank')
axes[1].set_title('(b) Oracle-free $M_4$')

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5, frameon=False,
           bbox_to_anchor=(0.5, -0.13))
fig.suptitle(f'RULER on Bio_ClinicalBERT ({DATA_SOURCE}); dashed lines mark the '
             f'nulls, N = {NUM_SEEDS} seeds', fontsize=8)
fig.tight_layout()

stem = os.path.join(FIGS_DIR, f'clinical_ruler_{DATA_SOURCE.lower().replace("-", "_")}')
fig.savefig(stem + '.pdf', bbox_inches='tight')
fig.savefig(stem + '.png', bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved {stem}.pdf / .png')

## 12. LaTeX table

In [ ]:
def fmt_p(p):
    if pd.isna(p):
        return 'n/a'
    return r'$<0.001$' if p < 0.001 else f'${p:.3f}$'


lines = [
    r'\begin{table}[t]', r'\footnotesize', r'\centering',
    r'\caption{RULER on Bio\_ClinicalBERT (' + DATA_SOURCE + r' clinical text). '
    r'$M_2$ null $=0$; $M_4$ null $=0.50$. One-sample Wilcoxon signed-rank test, '
    r'$N = ' + str(NUM_SEEDS) + r'$ seeds; with this many seeds the smallest '
    r'attainable $p$ is ' + f'{MIN_P_AT_N5:.3f}' + r', so no condition reaches '
    r'significance and the estimates should be read directly. Pre-unlearning '
    r'$M_4$ near 0.50 indicates little representation-level memorisation prior '
    r'to unlearning.}',
    r'\label{tab:clinical_lm}',
    r'\setlength{\tabcolsep}{4pt}',
    r'\begin{tabular}{llccc}',
    r'\toprule',
    r'$\mathtt{ff}$ & Method & $M_2$ & $M_4$ & Pre-unlearning $M_4$ \\',
    r'\midrule',
]

for ff in FORGET_FRACTIONS:
    pre = pre_df.loc[pre_df['forget_fraction'] == ff, 'pre_unlearning_m4'].mean()
    lines.append(rf'\multicolumn{{5}}{{l}}{{\textbf{{ff $= {ff:.0%}$}}}} \\'.replace('%', r'\%'))
    for method in METHODS:
        m2v = mean_of('m2', ff, method)
        m4v = mean_of('m4', ff, method)
        lines.append(f'& {method} & ${m2v:+.5f}$ & ${m4v:.4f}$ & ${pre:.4f}$ ' + r'\\')
    if ff != FORGET_FRACTIONS[-1]:
        lines.append(r'\addlinespace')

lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
latex = '\n'.join(lines)
print(latex)

with open(os.path.join(FIGS_DIR, 'clinical_ruler_table.tex'), 'w') as handle:
    handle.write(latex)
print(f'\nSaved {FIGS_DIR}clinical_ruler_table.tex')

## 13. Validation

Structural checks on the results: is every condition populated, are the metrics
in range, and do the values behave as Section 5.5 describes?

**What this does not do.** It does not flag $M_2 < 0$ or $M_4 > 0.50$ as
"expected" and their opposites as errors. In the tabular experiments $M_2$ is
consistently negative, but Section 5.5 reports that on clinical text $M_2$
takes **both signs** across methods (−0.048 for NegGrad+ and +0.033 for
Fine-Tuning at ff = 5%) and $M_4$ sits near 0.50. Asserting the tabular
directions here would flag the paper's own reported result as a failure. The
checks below verify *validity*, and the directional pattern is reported for
interpretation rather than judged.

In [ ]:
problems, observations = [], []

expected_rows = len(FORGET_FRACTIONS) * NUM_SEEDS * len(METHODS)
if len(results_df) != expected_rows:
    problems.append(f'expected {expected_rows} result rows, found {len(results_df)}')

for ff in FORGET_FRACTIONS:
    for method in METHODS:
        cell = results_df[(results_df['forget_fraction'] == ff)
                          & (results_df['method'] == method)]
        if len(cell) != NUM_SEEDS:
            problems.append(f'ff={ff:.0%} {method}: {len(cell)} seeds, expected {NUM_SEEDS}')

# Range checks. These catch genuine implementation faults: M4 is a proportion,
# and M2 is a difference of cosine similarities.
if not results_df.empty:
    if not results_df['m4'].between(0.0, 1.0).all():
        problems.append('M4 outside [0, 1] -- percentile rank is a proportion')
    if not results_df['m2'].between(-2.0, 2.0).all():
        problems.append('M2 outside [-2, 2] -- impossible for a cosine difference')
    if results_df[['m2', 'm4']].isna().any().any():
        problems.append('NaN values present in M2 or M4')

# Descriptive pattern, reported not judged.
for ff in FORGET_FRACTIONS:
    cell = results_df[results_df['forget_fraction'] == ff]
    if cell.empty:
        continue
    n_negative = int((cell['m2'] < 0).sum())
    pre = pre_df.loc[pre_df['forget_fraction'] == ff, 'pre_unlearning_m4'].mean()
    observations.append(
        f'ff={ff:.0%}: M2 negative in {n_negative}/{len(cell)} runs '
        f'(mean {cell["m2"].mean():+.5f}); M4 mean {cell["m4"].mean():.4f}; '
        f'pre-unlearning M4 {pre:.4f}')

print('VALIDATION')
print('=' * 62)
if problems:
    for item in problems:
        print(f'  [!] {item}')
else:
    print('  All structural checks passed.')

print('\nOBSERVED PATTERN (descriptive, not pass/fail)')
print('=' * 62)
for item in observations:
    print(f'  {item}')

print('\nSection 5.5 expects pre-unlearning M4 near 0.50 and M2 of both signs on')
print('this dataset -- unlike the tabular setting, where M2 is consistently')
print('negative. Weak memorisation means there is little to erase.')